In [ ]:
!pip install transformers accelerate torch torchvision gtts moviepy opencv-python bitsandbytes

from transformers import LlavaForConditionalGeneration, LlavaProcessor, pipeline
from moviepy.editor import VideoFileClip, AudioFileClip
from moviepy.editor import CompositeAudioClip
from gtts import gTTS
import torch
import cv2
import os
from PIL import Image
import shutil

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import numpy as np

model_id = "may-ur08/llava-commentary-gen"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    device_map="auto",          # Automatically use GPU/CPU efficiently
    torch_dtype=torch.float16,  # Keep FP16 for model precision
    load_in_4bit=True           # ✅ 4-bit quantization using bitsandbytes
)

In [33]:
video_path = "4.mp4"  # Your cricket video
video = VideoFileClip(video_path)

# Save video without audio
video_no_audio = video.without_audio()
video_no_audio.write_videofile("video_no_audio.mp4", codec="libx264")

# Extract 1 frame per second
fps = 4
frames = []
frame_dir = "frames"
os.makedirs(frame_dir, exist_ok=True)

cap = cv2.VideoCapture("video_no_audio.mp4")
count, saved = 0, 0
success = True

while success:
    success, image = cap.read()
    if success and int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % int(cap.get(cv2.CAP_PROP_FPS)) == 0:
        frame_path = f"{frame_dir}/frame_{saved}.jpg"
        cv2.imwrite(frame_path, image)
        frames.append(frame_path)
        saved += 1
cap.release()

Moviepy - Building video video_no_audio.mp4.
Moviepy - Writing video video_no_audio.mp4



t:  99%|█████████▊| 390/395 [00:06<00:00, 56.41it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.11/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file 4.mp4, 976896 bytes wanted but 0 bytes read,at frame 394/395, at time 8.04/8.06 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+



Moviepy - Done !
Moviepy - video ready video_no_audio.mp4


In [34]:
commentaries = []

for i, frame_path in enumerate(frames):
    image = Image.open(frame_path).convert("RGB")

    prompt = "<image>\nUSER: What cricket shot is being played in this image?\nASSISTANT:"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()

    # Remove repeated prompt
    if "ASSISTANT:" in caption:
        caption = caption.split("ASSISTANT:")[-1].strip()

    print(f"🎙️ Commentary [{i}]:", caption)
    commentaries.append(caption)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [0]: The batter in the image is playing a high drive shot.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [1]: The batter is playing a lofted shot.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

🎙️ Commentary [4]: The batter is playing a lofted shot.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [5]: The batter is playing a lofted shot.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [6]: The batter is playing a lofted shot.
🎙️ Commentary [7]: The batter is playing a sweep shot.


In [37]:
commentaries = []

for i, frame_path in enumerate(frames):
    image = Image.open(frame_path).convert("RGB")

    prompt = (
          "<image>\n"
    "USER: This is a single frame from a live cricket match broadcast. "
    "Analyze the batter’s footwork, body position, bat swing, and direction of contact. "
    "Identify the exact type of cricket shot being played (e.g., sweep, pull, cover drive, hook, cut, lofted shot, etc.). "
    "If the ball is hit low and timed well, assume it's a four. "
    "If the shot is aggressive and airborne, assume it's a six. "
    "If it clearly results in a dismissal, say it's out. "
    "If the result is not visible, say the outcome is uncertain. "
    "Only use proper cricket terminology — avoid any football, baseball, or non-cricket references. "
    "Now write a short, exciting cricket-style commentary line as if it's being broadcast on TV.\n"
    "ASSISTANT:" )
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=100)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()

    # Remove repeated prompt
    if "ASSISTANT:" in caption:
        caption = caption.split("ASSISTANT:")[-1].strip()

    print(f"🎙️ Commentary [{i}]:", caption)
    commentaries.append(caption)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [0]: "Spot on! The batter's footwork is sharp and balanced, the body position is strong, and the bat swing is fluid. The ball is hit with precision and power, sending it soaring through the air. It's a four! The batter's confidence is on display, and the crowd is roaring with excitement. This is a moment of pure cricket gold!"


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [1]: [Exciting cricket commentary] Ladies and gentlemen, welcome to this thrilling cricket match between India and New Zealand. We're in the midst of a high-stakes moment, with the ball coming towards the batter. The batter's footwork is swift and balanced, his body position is compact and ready, and his bat swing is smooth and controlled. The ball is hit with precision and power, soaring high into the air. It's a beautiful shot,


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

🎙️ Commentary [4]: [Exciting cricket commentary] Ladies and gentlemen, welcome to this thrilling cricket match between India and New Zealand. The batter, donned in a crisp white jersey and a focused expression, is at the crease, ready to take on the incoming ball. With a swift and calculated footwork, he strides towards the ball, his body positioning a study in concentration and readiness. As he swings his bat, he makes contact with the ball, sending


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [5]: [Excitedly] That's a fantastic shot! The batter's footwork was spot on, and the body position was perfect for a powerful sweep. The bat swung through the ball with precision, making contact at the highest point possible. The direction of contact was spot on, resulting in a four. The ball was hit low and timed perfectly, resulting in a six. The outcome is out!


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [6]: "And there we have it! The batter's footwork is impeccable, his body position is spot on, and the bat swing is a study in precision. The ball is hit with a sweet spot, and it's a four! The bowler's delivery was a bit wide, but the batter made it look easy. This is a classic example of how a well-timed swing can result in a beautiful shot. And there you have it, a four in
🎙️ Commentary [7]: "And there we have it! The batter's footwork is impeccable, and the body position is spot on. The bat swing is a study in precision, and the direction of contact is spot on. It's a beautiful sweep shot, executed with finesse. And the timing is perfect, resulting in a four. The ball is hit low and timed well, and the batter's confidence soars. It's a masterclass in cricket technique."


In [46]:
commentaries = []

for i, frame_path in enumerate(frames):
    image = Image.open(frame_path).convert("RGB")

    prompt = ("<image>\n"
    "USER: Analyze this image from a live cricket match.\n"
    "Identify two things:\n"
    "1. What specific type of cricket shot is being played?\n"
    "2. What is the likely outcome?\n"
    "Only use proper cricket terminology — avoid any football, baseball, or non-cricket references. "
    "Now write a short, exciting cricket-style commentary line as if it's being broadcast on TV.\n"
    "ASSISTANT:")
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()

    # Remove repeated prompt
    if "ASSISTANT:" in caption:
        caption = caption.split("ASSISTANT:")[-1].strip()

    print(f"🎙️ Commentary [{i}]:", caption)
    commentaries.append(caption)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [0]: "Spot on! That's a beautiful lofted shot from the batter. Great timing and power. The ball is flying high and straight, making it a perfect catch for the fielder. Out!"


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [1]: [Background] The ball is coming down at


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

🎙️ Commentary [4]: [Background noise]
COMMENTATOR: [Excited voice] "And there we have it! The batter plays a beautiful sweep shot, sending the ball soaring across the field. The bowler's delivery was spot on, but


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [5]: [Background noise]
CRICKET ANALYST: [Excited voice] "Ah, a classic shot! The batter has executed a beautiful sweep shot, sending the ball soaring across the field. The bowler's delivery


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🎙️ Commentary [6]: "And there we have it! The batter plays a beautiful sweep shot, sending the ball soaring across the field. The bowler's delivery was spot on, but the batter's timing was perfect. The ball lands in the gap between
🎙️ Commentary [7]: "And there we have it! The batter plays a beautiful sweep shot, sending the ball soaring across the field. The bowler's delivery was spot on, but the batter's timing was perfect. The ball is caught by the f


In [51]:
from transformers import pipeline

# Filter out noisy lines from commentaries
meaningful_lines = [
    c for c in commentaries
    if len(c.strip()) > 10 and
       "please provide" not in c.lower() and
       "frame" not in c.lower() and
       not c.lower().startswith("the teams playing") and
       not c.lower().startswith("assistant") and
       not c.strip().endswith(":")
]

# Join the valid commentary lines
joined_commentary = " ".join(meaningful_lines).strip()

# Fallback if nothing is usable
if not joined_commentary:
    joined_commentary = decoded  # fallback

# Load summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Generate summary directly on clean joined input
summary_output = summarizer(joined_commentary, max_length=30, min_length=20, do_sample=False)

# Extract final summarized line
llm_summary = summary_output[0]['summary_text']

print("\n🏏 Final Commentary Summary (LLM):", llm_summary)

Device set to use cuda:0



🏏 Final Commentary Summary (LLM): "Spot on! That's a beautiful lofted shot from the batter. Great timing and power. The ball is flying high and straight


In [52]:
tts = gTTS(text=llm_summary)
audio_path = "final_commentary.mp3"
tts.save(audio_path)

In [53]:
final_video_path = "final_video_with_commentary.mp4"

video = VideoFileClip(video_path).without_audio()
audio = AudioFileClip(audio_path)

# Adjust duration if needed
if audio.duration > video.duration:
    audio = audio.subclip(0, video.duration)

final_video = video.set_audio(audio)
final_video.write_videofile(final_video_path, codec="libx264", audio_codec="aac")

Moviepy - Building video final_video_with_commentary.mp4.
MoviePy - Writing audio in final_video_with_commentaryTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video final_video_with_commentary.mp4



t: 100%|██████████| 395/395 [00:07<00:00, 54.90it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.11/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file 4.mp4, 976896 bytes wanted but 0 bytes read,at frame 394/395, at time 8.04/8.06 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+



Moviepy - Done !
Moviepy - video ready final_video_with_commentary.mp4
